# 01 — What the published pipeline actually is

Before critiquing "the conventional pipeline", measure it. A pre-registered survey
(`survey/PROTOCOL.md`, committed before any data was collected) sampled public GitHub
notebooks that train a classifier on the Kaggle Heart Failure CSV and coded each against a
fixed rubric. Deviations are listed in `survey/DEVIATIONS.md`. No notebook or author is
named here; URLs and commit SHAs are in `survey/screening.csv`.

In [1]:
import json
import pandas as pd
from heart_audit.data import PROJECT_ROOT
from heart_audit.plots import survey_practices

S = PROJECT_ROOT / "survey"
screening = pd.read_csv(S / "screening.csv")
coding = pd.read_csv(S / "coding.csv", keep_default_na=False, na_values=[""])
summary = json.loads((S / "modal_pipeline.json").read_text(encoding="utf-8"))
snapshot = json.loads((S / "snapshot.json").read_text(encoding="utf-8"))
n = len(coding)
print(f"search matches: {snapshot['total_count']}, results frozen: {len(snapshot['items'])}")
print(f"screened: {len(screening)}, included: {n}")
screening.criterion.value_counts()

search matches: 3616, results frozen: 1000
screened: 57, included: 30


criterion
E2    11
E3     5
E1     5
E4     4
Name: count, dtype: int64

## Screening and reliability

In [2]:
double = pd.read_csv(S / "double_coding.csv", keep_default_na=False)
print(f"double-coded notebooks: {double.file.nunique()}, field agreement: {double.agree.sum()}/{len(double)}")
double[~double.agree][["file", "field", "first", "second", "resolved"]]

double-coded notebooks: 10, field agreement: 249/250


,file,field,first,second,resolved
131,041-1,stratify,yes,na,yes


Both coders are instances of the same language model working from the same protocol, so
this agreement measures how clear the rubric is and how consistently it is applied. It is not
agreement between independent human judges.

## Practices

In [3]:
practices = {
    "Evaluated on a single train/test split": (coding.eval_design == "single_split").sum(),
    "Scaler, imputer or other transform fit before the split": (coding.fit_before_split == "yes").sum(),
    "Split not stratified": (coding.stratify == "no").sum(),
    "Cholesterol = 0 kept as a real measurement": (coding.chol_zero == "ignored").sum(),
    "Duplicates not addressed": (coding.dedup == "none").sum(),
    "Best of several models reported on one test set": (coding.best_of_n == "yes").sum(),
    "Mentions ST_Slope was imputed at the source": (coding.source_imputation_mentioned == "yes").sum(),
}
survey_practices(list(practices), [int(v) for v in practices.values()], n,
                 PROJECT_ROOT / "images" / "survey_practices.png")
pd.Series(practices, name=f"of {n}")

Evaluated on a single train/test split                     27
Scaler, imputer or other transform fit before the split    19
Split not stratified                                       23
Cholesterol = 0 kept as a real measurement                 23
Duplicates not addressed                                   26
Best of several models reported on one test set            10
Mentions ST_Slope was imputed at the source                 0
Name: of 30, dtype: int64

![](../images/survey_practices.png)

## The modal pipeline (defines the control in notebook 02)

In [4]:
modal = pd.Series(summary["primary"]["modal"]).to_frame("modal value")
modal["share"] = pd.Series(summary["primary"]["shares"])
modal

,modal value,share
metric_name,accuracy,30/30
value_source,output,30/30
eval_design,single_split,27/30
test_size,0.2,23/27
stratify,no,23/30
seed_fixed,yes,26/30
models,"[decision_tree, logistic_regression, random_fo...",NaN
headline_model,random_forest,10/30
n_models,1.0,13/30
best_of_n,no,20/30


In [5]:
print("ties (primary):", summary["primary"]["ties"])
lit = summary["sensitivity_literal_e4"]["modal"]
print("fields that differ under the literal E4 reading:",
      [k for k in lit if lit[k] != summary["primary"]["modal"][k]])

ties (primary): []
fields that differ under the literal E4 reading: ['hyperparams']


The modal pipeline is assembled from per-field modes. It need not match any single
notebook exactly.

## Headline accuracy reported

In [6]:
pd.DataFrame({
    "primary": summary["primary"]["accuracy"],
    "literal E4 (sensitivity)": summary["sensitivity_literal_e4"]["accuracy"],
    "D2 best shown (sensitivity)": summary["sensitivity_d2_best_shown_accuracy"],
}).T

,n,median,iqr,min,max
primary,30,0.861,"[0.8205, 0.8803]",0.6961,0.9058
literal E4 (sensitivity),30,0.8632,"[0.8322, 0.8788]",0.6961,0.9058
D2 best shown (sensitivity),30,0.8658,"[0.8587, 0.8804]",0.6961,0.9444


## Limitations

- GitHub code search returns at most 1,000 results, ranked by relevance, and indexes only
  files under about 384 KB. The snapshot is therefore not a uniform sample of all matching
  notebooks.
- Coding was done from the notebooks' saved outputs, not by re-running them.
- Deviations D1–D4 (`survey/DEVIATIONS.md`) each have a stated sensitivity analysis or scope.